In [ ]:
import pandas as pd
import itertools
import random
from collections import defaultdict

In [ ]:
INPUT_DIR = './data'
df = pd.read_json(f'{INPUT_DIR}/final_dataset_gt.jsonl', lines=True)
sub_df = pd.read_json(f'{INPUT_DIR}/submissions.jsonl', lines=True)
df['id'] = df['sub_id'].str.rsplit('_', n=1).str[0]

df = df.merge(
    sub_df[['code', 'lang']],
    left_on='sub_id',
    right_on='sub_id',
    how='left'
)

In [ ]:
pairwise_records = []
criteria_cols = ['efficiency', 'correctness', 'readability']
MAX_APPEARANCES = 5 

for problem_id, group in df.groupby('id'):
    submissions = group.to_dict('records')
    
    # BƯỚC A: Tạo và xáo trộn tất cả các tổ hợp có thể có của problem này
    all_possible_pairs = list(itertools.combinations(submissions, 2))
    random.shuffle(all_possible_pairs)
    
    # Bộ đếm dùng chung để chốt cặp
    usage_count = defaultdict(int)
    selected_pairs = []
    
    # BƯỚC B: Chọn ra các cặp thỏa mãn giới hạn (chọn chung, không phụ thuộc criteria)
    for sub1, sub2 in all_possible_pairs:
        id1, id2 = sub1['sub_id'], sub2['sub_id']
        if usage_count[id1] < MAX_APPEARANCES and usage_count[id2] < MAX_APPEARANCES:
            # Ghi nhận cặp này đã được chọn
            usage_count[id1] += 1
            usage_count[id2] += 1
            selected_pairs.append((sub1, sub2))
            
    # BƯỚC C: Duyệt qua các cặp ĐÃ CHỐT để tạo dữ liệu cho từng criteria
    for sub1, sub2 in selected_pairs:
        for criteria in criteria_cols:
            score_col = f'{criteria}_score'
            score1 = sub1[score_col]
            score2 = sub2[score_col]
            
            # Chỉ tạo record nếu điểm của criteria này KHÁC NHAU
            if score1 != score2:
                label = 1 if score1 > score2 else 2
                
                pairwise_records.append({
                    'problem': problem_id,
                    'criteria': criteria,
                    'code1': sub1['code'],
                    'code2': sub2['code'],
                    'lang1': sub1['lang'],
                    'lang2': sub2['lang'],
                    'label': label
                })

pairwise_df = pd.DataFrame(pairwise_records)
pairwise_df